# 6.1 TD(0): 한 스텝만 보고 갱신한다 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter06_1_td0_bootstrap.ipynb)

책 본문: [6.1 TD(0): 한 스텝만 보고 갱신한다, TD vs MC](https://smhanlab.com/book-ml/kor/ml2/chapter06/1.html)

이 노트북은 6.1절의 내용을 코드로 직접 확인합니다:
(1) 7-state random walk의 참값을 벨만방정식으로 구해 닫힌형
\(V^*(i)=i(6-i)\)와 비교, (2) 첫 에피소드 \(3\to4\to5\to6\)를
TD(0)로 손으로 갱신한 값을 코드와 대조, (3) 같은 경험을 MC로 처리한 것과
비교, (4) 1000에피소드 학습으로 **TD vs MC의 목표값 분산 차이**를
실측, (5) 학습률 \(\alpha\)의 효과를 봅니다.

In [1]:
import matplotlib
matplotlib.use("Agg")
from matplotlib import font_manager
import matplotlib.pyplot as plt
kr = [f.name for f in font_manager.fontManager.ttflist if "Noto Sans CJK KR" in f.name]
if kr: plt.rcParams["font.sans-serif"] = [kr[0]]
plt.rcParams["axes.unicode_minus"] = False
import numpy as np
import random
IMG = "/home/smhan/book-ml/kor/src/images"


## 1. 7-state random walk 환경과 참값

상태 0~6이 일렬로, 매 스텝 확률 1/2씩 좌우 이동. 0과 6은 **터미널**(에피소드
종료), **매 스텝 \(+1\)** 보상, \(\gamma=1\). 참값
\(V^*(i)=i(6-i)\)(기대 흡수 시간)이 벨만방정식
\(V_i = 1 + \tfrac{1}{2}V_{i-1} + \tfrac{1}{2}V_{i+1}\)의 해인지
반복으로 직접 확인합니다.

In [2]:
TERMINALS = (0, 6)
N_STATES = 7

def step(s, rng):
    if s in TERMINALS:
        return s, 0.0
    return s + rng.choice([-1, 1]), 1.0   # 매 스텝 +1

# 참값: 벨만방정식 반복
Vstar = [0.0] * N_STATES
for _ in range(100000):
    for s in range(1, 6):
        Vstar[s] = 1.0 + 0.5 * Vstar[s - 1] + 0.5 * Vstar[s + 1]
closed = [i * (6 - i) for i in range(N_STATES)]
print("V* (벨만 반복):  ", np.round(Vstar, 3))
print("닫힌형 i(6-i):   ", closed)
print("일치:", np.allclose(Vstar, closed, atol=1e-6))

V* (벨만 반복):   [0. 5. 8. 9. 8. 5. 0.]
닫힌형 i(6-i):    [0, 5, 8, 9, 8, 5, 0]
일치: True


## 2. 첫 에피소드 \(3\to4\to5\to6\)를 TD(0)로 손갱신

\(\alpha=0.5\), \(\gamma=1\), 초기값 0. 본문과 같이 스텝마다
목표값 \(r + \gamma V(s')\)과 TD 오차를 표시합니다. 같은 에피소드를
MC(매방문)로 처리하면 어떻게 되는지도 비교합니다.

In [3]:
V = [0.0] * N_STATES
traj = [(3, 4), (4, 5), (5, 6)]
print("[TD(0)] 첫 에피소드 3->4->5->6, alpha=0.5, gamma=1:")
for s, sp in traj:
    r = 1.0
    target = r + (0.0 if sp in TERMINALS else V[sp])
    delta = target - V[s]
    V[s] += 0.5 * delta
    print(f"  s={s}->{sp}: 목표={target:.2f}, TD오차={delta:+.2f}, V({s})={V[s]:.2f}")
print("V =", [round(v, 2) for v in V])
print("다음에 4가 방문되면 목표값 =", 1.0 + V[5],
      " (첫 에피소드 당시 1.00에서 상승 — 정보 전파)")

# 같은 경험을 MC(매방문, 리턴 G=+3)로
V_mc = [0.0] * N_STATES
G = 3.0
for s in (3, 4, 5):
    V_mc[s] = G
print("\n[MC] 같은 에피소드: 리턴 G =", G, " -> V(3)=V(4)=V(5) =", V_mc[3])

[TD(0)] 첫 에피소드 3->4->5->6, alpha=0.5, gamma=1:
  s=3->4: 목표=1.00, TD오차=+1.00, V(3)=0.50
  s=4->5: 목표=1.00, TD오차=+1.00, V(4)=0.50
  s=5->6: 목표=1.00, TD오차=+1.00, V(5)=0.50
V = [0.0, 0.0, 0.0, 0.5, 0.5, 0.5, 0.0]
다음에 4가 방문되면 목표값 = 1.5  (첫 에피소드 당시 1.00에서 상승 — 정보 전파)

[MC] 같은 에피소드: 리턴 G = 3.0  -> V(3)=V(4)=V(5) = 3.0


## 3. 1000에피소드 학습: TD(0) vs MC (시드 42)

두 알고리즘을 같은 시드로 돌려 \(V(3)\)의 수렴을 비교합니다.
참값 \(V^*(3)=9\)에 둘 다 도달하지만 경로가 다릅니다 — TD는
**매 스텝** 갱신(덜 정확하지만 작은 갱신), MC는 **에피소드 끝**에
한 번 갱신(큰 점프).

In [4]:
def td0_train(seed=42, n_episodes=1000, alpha=0.1, start=3):
    rng = random.Random(seed)
    V = [0.0] * N_STATES
    history = []
    for ep in range(n_episodes):
        s = start
        while s not in TERMINALS:
            sp, r = step(s, rng)
            V[s] += alpha * (r + (0.0 if sp in TERMINALS else V[sp]) - V[s])
            s = sp
        if (ep + 1) % 10 == 0:
            history.append((ep + 1, list(V)))
    return V, history

def mc_train(seed=42, n_episodes=1000, start=3):
    rng = random.Random(seed)
    sums = [0.0] * N_STATES
    cnt = [0] * N_STATES
    history = []
    for ep in range(n_episodes):
        s = start
        states, rs = [], []
        while s not in TERMINALS:
            sp, r = step(s, rng)
            states.append(s); rs.append(r); s = sp
        G = 0.0
        for t in range(len(states) - 1, -1, -1):   # 뒤에서 리턴 누적
            G = rs[t] + G
            sums[states[t]] += G
            cnt[states[t]] += 1
        if (ep + 1) % 10 == 0:
            history.append((ep + 1, [sums[s]/cnt[s] if cnt[s] else 0.0 for s in range(N_STATES)]))
    return [sums[s]/cnt[s] if cnt[s] else 0.0 for s in range(N_STATES)], history

V_td, td_hist = td0_train(alpha=0.1)
V_m,  mc_hist = mc_train()
print("참값         =", [round(v, 1) for v in Vstar])
print("TD(0) a=0.1:", [round(v, 2) for v in V_td])
print("MC 매방문   :", [round(v, 2) for v in V_m])

참값         = [0.0, 5.0, 8.0, 9.0, 8.0, 5.0, 0.0]
TD(0) a=0.1: [0.0, 4.8, 8.52, 9.98, 9.43, 4.99, 0.0]
MC 매방문   : [0.0, 5.34, 8.33, 9.26, 8.35, 5.52, 0.0]


In [5]:
ep_td = [e for e, _ in td_hist]; V3_td = [V[3] for _, V in td_hist]
ep_mc = [e for e, _ in mc_hist]; V3_mc = [V[3] for _, V in mc_hist]
fig, ax = plt.subplots(figsize=(8.5, 4.5))
ax.plot(ep_mc, V3_mc, lw=1.0, alpha=0.55, color="#1a3d7c", label="MC (every visit)")
ax.plot(ep_td, V3_td, lw=1.4, color="#9c2b1e", label="TD(0), $\\alpha$=0.1")
ax.axhline(Vstar[3], color="green", ls=":", lw=1.8, label=f"true V*(3) = {Vstar[3]:.0f}")
ax.set_xlabel("Episodes"); ax.set_ylabel("V(3) estimate")
ax.set_title("7-state random walk: convergence of the V(3) estimate (seed 42)")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(IMG + "/ch06_1_td0_mc_v3.svg", bbox_inches="tight")
plt.show()

## 4. 핵심: 목표값 분산 비교 — "TD의 분산이 작다"의 실측

본문의 표에서 "TD의 분산이 작다"는 **갱신 한 번에 쓰는 목표값**의
산포 이야기를 의미합니다. MC의 목표값은 리턴 \(G\)(에피소드 **전체**
의 운 — 3스텝~50스텝 이상), TD의 목표값은 \(1 + V(s')\)(한 스텝의
보상 + 현재 추정치)입니다. 1000에피소드(시드 42)에서 실제로 수집해
그림으로 비교합니다.

In [6]:
Gs = []   # 각 에피소드의 실제 리턴 G (출발 상태 3 기준, 에피소드 전체)
rng = random.Random(42)
for _ in range(1000):
    s = 3
    G = 0.0
    while s not in TERMINALS:
        sp, r = step(s, rng)
        G += r
        s = sp
    Gs.append(G)

# TD 목표값: 같은 시드로 TD(0)를 돌리며 각 스텝의 목표값 기록
td_targets = []
rng2 = random.Random(42)
Vt = [0.0] * N_STATES
for _ in range(1000):
    s = 3
    while s not in TERMINALS:
        sp, r = step(s, rng2)
        tgt = r + (0.0 if sp in TERMINALS else Vt[sp])
        td_targets.append(tgt)
        Vt[s] += 0.1 * (tgt - Vt[s])
        s = sp

Garr, Tarr = np.array(Gs), np.array(td_targets)
print(f"MC 목표값(리턴 G)  : 평균={Garr.mean():.2f}  표준편차={Garr.std():.2f}  min={Garr.min():.0f}  max={Garr.max():.0f}")
print(f"TD 목표값(1+V(s')) : 평균={Tarr.mean():.2f}  표준편차={Tarr.std():.2f}  min={Tarr.min():.2f}  max={Tarr.max():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].hist(Garr, bins=12, color="#1a3d7c", alpha=0.85, edgecolor="white")
axes[0].axvline(Vstar[3], color="red", ls="--", lw=1.6, label=f"true V*(3) = {Vstar[3]:.0f}")
axes[0].set_title("MC target = actual return G\navg %.1f · std %.1f · range %d-%d" % (Garr.mean(), Garr.std(), Garr.min(), Garr.max()))
axes[0].set_xlabel("Target"); axes[0].set_ylabel("Frequency"); axes[0].legend()
axes[1].hist(Tarr, bins=12, color="#9c2b1e", alpha=0.85, edgecolor="white")
axes[1].axvline(Vstar[3], color="red", ls="--", lw=1.6, label=f"true V*(3) = {Vstar[3]:.0f}")
axes[1].set_title("TD(0) target = r + $\\gamma$ V(s')\navg %.1f · std %.1f · range %.1f-%.1f" % (Tarr.mean(), Tarr.std(), Tarr.min(), Tarr.max()))
axes[1].set_xlabel("Target"); axes[1].set_ylabel("Frequency"); axes[1].legend()
fig.suptitle("TD vs MC: spread of the target used per update — MC draws the luck of the whole episode, TD only one step", y=1.03)
fig.tight_layout()
fig.savefig(IMG + "/ch06_1_td0_target_var.svg", bbox_inches="tight")
plt.show()

MC 목표값(리턴 G)  : 평균=9.32  표준편차=7.28  min=3  max=51
TD 목표값(1+V(s')) : 평균=7.63  표준편차=2.98  min=1.00  max=11.69


## 5. 학습률 \(\alpha\)가 크면 참값 주위를 맴돈다

"수렴하려면 \(\alpha\)가 줄어야 한다"(Robbins-Monro)는 조건을
직관적으로 확인합니다. \(\alpha\)가 크면 잡음이 섞인 목표값이
**통째** 들어와 참값 9 주위를 크게 맴돕니다(고정 \(\alpha\)의 TD는
참값에 정확히 "착지"하지 않는다 — 본문 "자주 하는 실수" 참고).

In [7]:
fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.8), sharey=True)
for ax, alpha in zip(axes, [0.5, 0.1, 0.01]):
    Vt, h = td0_train(alpha=alpha)
    ax.plot([e for e, _ in h], [V[3] for _, V in h], lw=0.9, color="#9c2b1e")
    ax.axhline(Vstar[3], color="green", ls=":", lw=1.5)
    tail = [V[3] for _, V in h[-10:]]
    ax.set_title("$\\alpha$ = %s (mean of last 10 points: %.2f)" % (alpha, np.mean(tail)))
    ax.set_xlabel("Episode"); ax.grid(alpha=0.3)
axes[0].set_ylabel("V(3) estimate")
fig.suptitle("Larger learning rate $\\alpha$: target noise feeds directly into the update, so V(3) swings widely around the true value (9) (seed 42, 1000 episodes)", y=1.03)
fig.tight_layout()
fig.savefig(IMG + "/ch06_1_td0_alpha.svg", bbox_inches="tight")
plt.show()

## 6. 정리

- **부트스트래핑**: TD(0)의 목표값 \(r+\gamma V(s')\)는 "에피소드가
  끝날 때"가 아니라 "한 스텝 뒤의 **현재 추정치**"를 미래 가치로
  대입한다 — 그래서 에피소드가 끝나기 전에도 매 스텝 학습이
  가능하다(Section 2의 4→5 스텝: 에피소드 중반인데도 V(4) 갱신).
- **편향-분산 트레이드오프**: MC의 목표값은 리턴(운을 통째로 담고,
  큰 분산), TD의 목표값은 한 스텝(작은 분산, 추정치가 부정확한 동안
  편향)이다 — Section 4에서 실측.
- **수렴**: TD 오차가 0이 되는 지점이 벨만방정식의 해
  \(V^\pi\)이며, \(\alpha\)가 적절히 줄어갈 때 그 해로
  수렴한다(Section 3, 5).